# H3-3 — Cryptanalyse par transposition

**Objectif** : casser un chiffrement par transposition columnar avec CP-SAT.

## Plan
1. [Le chiffrement par transposition](#1)
2. [Propriétés statistiques — IC conservé](#2)
3. [Modèle CP-SAT (longueur connue)](#3)
4. [Pipeline complet et benchmark](#4)
5. [Limites et variantes](#5)

In [2]:
import sys, os, time, random
import string
import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath('..'))

from core.ciphers.transposition     import generate_random_key, encrypt, decrypt, key_accuracy
from core.linguistics.frequency_analysis import (
    clean_text, letter_frequencies, bigram_log_probs,
    index_of_coincidence, score_text
)
from core.solvers.cp_transposition  import solve_transposition

ALPHABET = string.ascii_uppercase
random.seed(42)

In [3]:
with open(os.path.join('..', 'data', 'french_reference.txt'), encoding='utf-8') as f:
    CORPUS = f.read()

BIGRAM_LP   = bigram_log_probs(CORPUS)
LETTER_FREQ = letter_frequencies(CORPUS)
print(f"Corpus : {len(clean_text(CORPUS))} lettres")

Corpus : 8314 lettres


<a id='1'></a>
---
## 1. Le chiffrement par transposition

### Principe

Les lettres ne sont **pas substituées** — elles gardent leur valeur — mais **réarrangées** selon une permutation de colonnes.

```
Texte clair : BONJOURPARIS  (12 lettres, L=4)

Matrice (4 colonnes) :
  col: 0 1 2 3
       B O N J
       O U R P
       A R I S

Clé = [3, 1, 0, 2]  (lire les colonnes dans cet ordre)
  → seg 0: col 3 → J P S
  → seg 1: col 1 → O U R
  → seg 2: col 0 → B O A
  → seg 3: col 2 → N R I

Chiffré : JPSOURBOANRI
```

**Espace** : L! permutations — 4! = 24, 6! = 720, 8! = 40 320.  
Beaucoup plus petit que la substitution (26! ≈ 4×10²⁶), mais les fréquences sont  
**identiques** entre clair et chiffré.

In [ ]:
# ── Texte de test ─────────────────────────────────────────────────────────
TEST_PLAIN = clean_text("""
    La justice est lame de la societe comme le feu est lame des forges.
    On ne peut pas vivre sans justice comme on ne peut pas vivre sans pain.
    Il faut regarder en face la misere pour la comprendre et pour la vaincre.
    Jean Valjean etait un homme de taille forte et robuste dans la fleur de l age.
    Paris est une ville immense ou se melent toutes les races et toutes les fortunes.
    La France est un grand pays avec une longue histoire et une culture riche.
""")

KEY_LENGTH = 6
TRUE_KEY   = generate_random_key(KEY_LENGTH)
CIPHERTEXT = encrypt(TEST_PLAIN, TRUE_KEY)

print(f"Texte clair  : {len(TEST_PLAIN)} lettres")
print(f"Clé          : {TRUE_KEY}  (L={KEY_LENGTH})")
print(f"Texte chiffré (extrait): {CIPHERTEXT[:80]}")

# Vérification round-trip (decrypt retourne le texte paddé, on compare les N premières lettres)
assert decrypt(CIPHERTEXT, TRUE_KEY)[:len(TEST_PLAIN)] == TEST_PLAIN
print("\nencrypt/decrypt OK")

<a id='2'></a>
---
## 2. Propriétés statistiques — IC conservé

La transposition **préserve les fréquences de lettres** (et donc l'IC),  
contrairement au Vigenère qui les aplatit.

```
IC(transposition) ≈ IC(clair) ≈ 0.065  (français)
IC(Vigenère)      ≈ 0.038–0.055
```

→ L'IC permet de **distinguer** les deux types de chiffrements.  
→ L'analyse de fréquence seule ne suffit pas (les fréquences sont identiques).  
→ Il faut attaquer les **bigrammes** : leur distribution est perturbée par la transposition.

In [ ]:
from core.ciphers.vigenere import encrypt as v_encrypt

cipher_transpo = CIPHERTEXT
cipher_vigenere = clean_text(v_encrypt(TEST_PLAIN, 'FRANCE'))

print(f"IC texte clair      : {index_of_coincidence(TEST_PLAIN):.4f}  (français ≈ 0.065)")
print(f"IC transposition    : {index_of_coincidence(cipher_transpo):.4f}  (conservé ≈ 0.065)")
print(f"IC Vigenère (FRANCE): {index_of_coincidence(cipher_vigenere):.4f}  (abaissé)")

# Comparaison des fréquences de lettres
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

def plot_freq(ax, freq_dict, title, color):
    vals = [freq_dict.get(l, 0) for l in ALPHABET]
    ax.bar(list(ALPHABET), vals, color=color, alpha=0.85)
    ax.set_title(title, fontsize=11)
    ax.tick_params(axis='x', labelsize=7)

plot_freq(axes[0], letter_frequencies(TEST_PLAIN),    'Texte clair',       'seagreen')
plot_freq(axes[1], letter_frequencies(cipher_transpo),'Transposition',      'steelblue')
plot_freq(axes[2], letter_frequencies(cipher_vigenere),'Vigenère (FRANCE)', 'tomato')

plt.suptitle('Fréquences — transposition préserve les fréquences', y=1.02)
plt.tight_layout()
os.makedirs('../examples', exist_ok=True)
plt.savefig('../examples/transpo_freq.png', dpi=120, bbox_inches='tight')
plt.show()

# Score bigrammes
print(f"\nScore bigrammes clair      : {score_text(TEST_PLAIN, BIGRAM_LP):.0f}")
print(f"Score bigrammes transposition: {score_text(cipher_transpo, BIGRAM_LP):.0f}")
print("→ Les bigrammes sont perturbés → signal pour CP-SAT")

<a id='3'></a>
---
## 3. Modèle CP-SAT (longueur connue)

### Le CSP

| Élément | Description |
|---------|-------------|
| **Variables** | `pos[c]` pour c ∈ {0..L-1}, domaine [0..L-1] |
| | `pos[c] = j` ↔ colonne originale c → segment de sortie j |
| **Contrainte** | `AllDifferent(pos)` — permutation |
| **Objectif** | Minimiser coût bigrammes des paires de colonnes consécutives |

### Table d'agrégation

Pour la colonne originale c lue à la position `pos[c] = p` et la colonne c+1 à `pos[c+1] = q` :

```python
# Bigrams within-row for columns c and c+1:
agg_table[p*L + q] = Σ_row  bigram_cost( cipher[p*n_rows+row], cipher[q*n_rows+row] )
```

Cette table est **unique** (même pour toutes les paires consécutives) → L−1 contraintes `AddElement`.

```
                    Espace : L! ≤ 40320 (L=8)
Contraintes CP-SAT : AllDifferent(pos) + (L-1) AddElement
Temps              : < 2s pour L ≤ 8 sur 400 lettres
```

In [ ]:
# ── CP-SAT avec vraie longueur ────────────────────────────────────────────
print(f"Résolution CP-SAT (L={KEY_LENGTH}, texte={len(TEST_PLAIN)} lettres)...")
t0 = time.time()
result = solve_transposition(CIPHERTEXT, KEY_LENGTH, BIGRAM_LP, time_limit=30.0)
elapsed = time.time() - t0

print(f"Statut     : {result['status']}  |  Temps: {elapsed:.2f}s")
print(f"Clé trouvée: {result['key']}")
print(f"Vraie clé  : {TRUE_KEY}")
print(f"Précision  : {key_accuracy(TRUE_KEY, result['key']):.0%}")

if result['plaintext']:
    plain_score  = score_text(TEST_PLAIN,         BIGRAM_LP)
    found_score  = score_text(result['plaintext'], BIGRAM_LP)
    cipher_score = score_text(CIPHERTEXT,          BIGRAM_LP)
    print(f"\nScore bigrammes clair     : {plain_score:.0f}")
    print(f"Score bigrammes trouvé    : {found_score:.0f}")
    print(f"Score bigrammes chiffré   : {cipher_score:.0f}")
    print(f"\nDéchiffré (extrait): {result['plaintext'][:100]}")
    print(f"Vrai texte (extrait): {TEST_PLAIN[:100]}")

In [ ]:
# ── Visualiser le score bigrammes selon la permutation ───────────────────
import itertools

# Pour une petite clé (L=4), énumérer toutes les permutations et leurs scores
DEMO_L = 4
demo_key = generate_random_key(DEMO_L)
# Ajuster la longueur du texte pour être divisible par DEMO_L
demo_plain = (TEST_PLAIN * 2)[:200 - 200 % DEMO_L]
demo_cipher = encrypt(demo_plain, demo_key)

n_rows_demo = len(demo_cipher) // DEMO_L

all_perms = list(itertools.permutations(range(DEMO_L)))
perm_scores = []
for perm in all_perms:
    # perm = key
    plain_cand = decrypt(demo_cipher, list(perm))
    perm_scores.append(score_text(plain_cand, BIGRAM_LP))

true_perm_idx = all_perms.index(tuple(demo_key))
best_perm_idx = int(np.argmax(perm_scores))

plt.figure(figsize=(12, 4))
plt.bar(range(len(all_perms)), perm_scores, color='steelblue', alpha=0.6)
plt.axvline(true_perm_idx,  color='seagreen', linewidth=2, label=f'Vraie clé {demo_key}  score={perm_scores[true_perm_idx]:.0f}')
plt.axvline(best_perm_idx,  color='tomato',   linewidth=2, linestyle='--', label=f'Meilleure permu {list(all_perms[best_perm_idx])}  score={perm_scores[best_perm_idx]:.0f}')
plt.xlabel(f'Permutation (parmi {len(all_perms)} pour L=4)')
plt.ylabel('Score bigrammes log-vraisemblance')
plt.title(f'Score bigrammes de toutes les permutations (L=4, n={len(demo_cipher)} lettres)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig('../examples/transpo_perm_scores.png', dpi=120, bbox_inches='tight')
plt.show()

print(f"Vraie clé   : {demo_key}  (rang: {sorted(perm_scores, reverse=True).index(perm_scores[true_perm_idx])+1}/{len(all_perms)})")
print(f"Meilleure   : {list(all_perms[best_perm_idx])}")
print(f"Vrais et meilleurs identiques : {true_perm_idx == best_perm_idx}")

<a id='4'></a>
---
## 4. Pipeline complet et benchmark

On teste CP-SAT sur différentes longueurs de clé et de texte.

In [ ]:
KEY_LENGTHS  = [4, 5, 6, 7, 8]
TEXT_LENGTHS = [100, 200, 400]
N_TRIALS     = 3
CP_LIMIT     = 20.0

base_text = clean_text(CORPUS) * 3

bench_results = {}  # (key_len, text_len) → {acc, time}

for kl in KEY_LENGTHS:
    for tl in TEXT_LENGTHS:
        # Adjust text length to be divisible by key length
        tl_adj = tl - tl % kl
        if tl_adj == 0:
            continue
        plain_sample = base_text[:tl_adj]
        accs, times = [], []
        for trial in range(N_TRIALS):
            key = generate_random_key(kl)
            cipher = encrypt(plain_sample, key)
            t0 = time.time()
            res = solve_transposition(cipher, kl, BIGRAM_LP, time_limit=CP_LIMIT)
            elapsed = time.time() - t0
            acc = key_accuracy(key, res['key']) if res['key'] else 0.0
            accs.append(acc)
            times.append(elapsed)
        mean_acc  = np.mean(accs)
        mean_time = np.mean(times)
        bench_results[(kl, tl)] = {'acc': mean_acc, 'time': mean_time}
        print(f"L={kl:2d}  n={tl:4d}  acc={mean_acc:.0%}  t={mean_time:.1f}s")

In [ ]:
# ── Graphique : précision vs longueur de clé ─────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

colors = plt.cm.viridis(np.linspace(0.2, 0.9, len(TEXT_LENGTHS)))

for i, tl in enumerate(TEXT_LENGTHS):
    accs  = [bench_results.get((kl, tl), {}).get('acc',  0) for kl in KEY_LENGTHS]
    times = [bench_results.get((kl, tl), {}).get('time', 0) for kl in KEY_LENGTHS]
    ax1.plot(KEY_LENGTHS, [a*100 for a in accs],  'o-', color=colors[i], label=f'n={tl}', lw=2)
    ax2.plot(KEY_LENGTHS, times, 's-', color=colors[i], label=f'n={tl}', lw=2)

ax1.set_xlabel('Longueur de clé L')
ax1.set_ylabel('Précision clé (%)')
ax1.set_title('Précision CP-SAT vs longueur de clé')
ax1.set_ylim(0, 105)
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.set_xlabel('Longueur de clé L')
ax2.set_ylabel('Temps (s)')
ax2.set_title('Temps CP-SAT vs longueur de clé')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.suptitle('Benchmark — Transposition columnar (CP-SAT)', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('../examples/benchmark_transposition.png', dpi=120, bbox_inches='tight')
plt.show()

<a id='5'></a>
---
## 5. Limites et variantes

### Limites du modèle actuel

- Seuls les **bigrammes within-row** sont exploités (colonnes consécutives dans la même ligne)
- Les bigrammes **entre lignes** (fin de colonne c, début de colonne c du row suivant) ne sont pas modélisés
- Performance dégradée pour des textes courts (< 100 lettres)

### Comparaison avec Vigenère

| Aspect | Transposition | Vigenère |
|--------|--------------|----------|
| Espace de clé | L! | 26^L |
| IC chiffré | Identique au clair (~0.065) | Abaissé (~0.038-0.055) |
| Signal CP-SAT | Bigrammes within-row | Bigrammes agrégés par paire de positions |
| Complexité modèle | L-1 constraints | L² constraints |

### Amélioration : inclure les bigrammes between-rows

```python
# Bigram at positions (row*L + L-1) and ((row+1)*L + 0)
# plain[row*L + L-1] = cipher[pos[L-1]*n_rows + row]
# plain[(row+1)*L + 0] = cipher[pos[0]*n_rows + (row+1)]
# Cost depends on (pos[L-1], pos[0]) with DIFFERENT row offsets → separate aggregate table
```

Cette extension nécessite L tables supplémentaires avec des offsets de ligne différents.

In [ ]:
# ── Tableau récapitulatif ─────────────────────────────────────────────────
print(f"{'L':>4} │", end='')
for tl in TEXT_LENGTHS:
    print(f" n={tl:4d} Préc.  Temps │", end='')
print()
print('─' * (6 + len(TEXT_LENGTHS) * 22))
for kl in KEY_LENGTHS:
    print(f"{kl:>4} │", end='')
    for tl in TEXT_LENGTHS:
        r = bench_results.get((kl, tl))
        if r:
            print(f" {r['acc']:>9.0%}  {r['time']:>5.1f}s │", end='')
        else:
            print(f"       N/A        │", end='')
    print()

print("\n→ Suite : H3-4 — Hill cipher (chiffrement linéaire par blocs)")